In [1]:
import gc
import os
from datetime import date, timedelta

import catboost
import numpy as np
import pandas as pd
import polars as pl

In [2]:
test_start_date = date(2024, 8, 1)
val_start_date = date(2024, 7, 1)
val_end_date = date(2024, 7, 31)
train_end_date = date(2024, 6, 30)
data_path = './predict-user-fresh-order'

# Read data

In [3]:
actions_history = pl.scan_parquet(os.path.join(data_path, 'actions_history'))
search_history = pl.scan_parquet(os.path.join(data_path, 'search_history'))
product_information = pl.scan_csv(os.path.join(data_path, 'product_information.csv'))

In [4]:
product_information.head(100).collect(streaming=True)

/tmp/ipykernel_22168/3577795817.py:1: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  product_information.head(100).collect(streaming=True)


product_id,name,brand,type,category_id,category_name,price,discount_price
i64,str,str,str,i64,str,f64,f64
26176363,"""Развивающие тесты (3-4 года) (…","""Machaon""","""Печатная книга: Развитие детей""",780,"""Книги""",380.0,274.0
29898500,"""Mexx Туалетная вода Ice Touch …","""Mexx""","""Туалетная вода""",117,"""Мужская""",2645.0,1859.0
33967827,"""64 ГБ USB Флеш-накопитель USB …","""SmartBuy""","""USB-флеш-накопитель""",178,"""Флешки и CD-R""",1690.0,469.0
135938830,"""Чай листовой чёрный Ahmad Tea …","""Ahmad Tea""","""Чай листовой""",465,"""Чай листовой""",319.0,244.0
137920686,"""Seagate 4 ТБ Внешний жесткий д…","""Seagate""","""Внешний жесткий диск""",615,"""Внешние жесткие диски""",28590.0,9539.0
…,…,…,…,…,…,…,…
499019785,"""Набор для игры в покер ""Poker …","""Анзор Игра""","""Набор для покера""",5,"""Настольные игры для детей""",999.0,425.0
519410131,"""Органайзер для хранения / Орга…","""Baona""","""Органайзер для вещей""",37,"""Товары для дачи""",1490.0,838.0
524083710,"""Футляр для очков на магните му…","""O.Rissa""","""Футляр для очков""",56,"""Очки и аксессуары""",850.0,407.0


In [5]:
pl.read_csv(os.path.join(data_path, 'action_type_info.csv'))

action_type,action_type_id
str,i64
"""click""",1
"""favorite""",2
"""order""",3
"""search""",4
"""to_cart""",5
"""view""",6


In [6]:
val_start_date, val_end_date

(datetime.date(2024, 7, 1), datetime.date(2024, 7, 31))

In [7]:
val_target = (
    actions_history
    .filter(pl.col('timestamp').dt.date() >= val_start_date)
    .filter(pl.col('timestamp').dt.date() <= val_end_date)
    .select('user_id', (pl.col('action_type_id') == 3).alias('has_order'))
    .group_by('user_id')
    .agg(pl.max('has_order').cast(pl.Int8).alias('target'))
    .collect(streaming=True)
)

/tmp/ipykernel_22168/3786263777.py:8: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)


In [8]:
val_target.group_by('target').agg(pl.len().alias('num_users'))

target,num_users
i8,u32
0,1227381
1,647575


In [9]:
val_target.schema

Schema([('user_id', Int32), ('target', Int8)])

# Simple pipeline

## Feats

In [10]:
train_end_date, train_end_date - timedelta(days=30 * 4)

(datetime.date(2024, 6, 30), datetime.date(2024, 3, 2))

In [11]:
def build_baseline_features(users_df: pl.DataFrame, history_end_date: date, pred_start_date: date) -> pl.DataFrame:
    history_start_date = history_end_date - timedelta(days=30 * 4)

    product_prices = product_information.select('product_id', 'discount_price')

    df = users_df

    actions_id_to_suf = {
        1: "click",
        2: "favorite",
        3: "order",
        5: "to_cart",
    }

    for action_id, suf in actions_id_to_suf.items():
        feat = (
            actions_history
            .filter(pl.col('timestamp').dt.date() <= history_end_date)
            .filter(pl.col('timestamp').dt.date() >= history_start_date)
            .filter(pl.col('action_type_id') == action_id)
            .select(['user_id', 'product_id', 'timestamp'])
            .join(product_prices, on='product_id', how='left')
            .group_by('user_id')
            .agg(
                pl.len().cast(pl.Int32).alias(f'num_products_{suf}'),
                pl.sum('discount_price').cast(pl.Float32).alias(f'sum_discount_price_{suf}'),
                pl.max('discount_price').cast(pl.Float32).alias(f'max_discount_price_{suf}'),
                pl.max('timestamp').alias(f'last_{suf}_time'),
                pl.min('timestamp').alias(f'first_{suf}_time'),
            )
            .with_columns([
                (pl.lit(pred_start_date) - pl.col(f'last_{suf}_time'))
                .dt.total_days()
                .cast(pl.Int32)
                .alias(f'days_since_last_{suf}'),

                (pl.lit(pred_start_date) - pl.col(f'first_{suf}_time'))
                .dt.total_days()
                .cast(pl.Int32)
                .alias(f'days_since_first_{suf}'),
            ])
            .select(
                'user_id',
                f'num_products_{suf}',
                f'sum_discount_price_{suf}',
                f'max_discount_price_{suf}',
                f'days_since_last_{suf}',
                f'days_since_first_{suf}',
            )
            .collect(streaming=True)
        )

        df = df.join(feat, on='user_id', how='left')
        del feat
        gc.collect()

    search_feat = (
        search_history
        .filter(pl.col('action_type_id') == 4)
        .filter(pl.col('timestamp').dt.date() <= history_end_date)
        .filter(pl.col('timestamp').dt.date() >= history_start_date)
        .select(['user_id', 'search_query', 'timestamp'])
        .group_by('user_id')
        .agg(
            pl.len().cast(pl.Int32).alias('num_search'),
            pl.max('timestamp').alias('last_search_time'),
            pl.min('timestamp').alias('first_search_time'),
        )
        .with_columns([
            (pl.lit(pred_start_date) - pl.col('last_search_time'))
            .dt.total_days()
            .cast(pl.Int32)
            .alias('days_since_last_search'),

            (pl.lit(pred_start_date) - pl.col('first_search_time'))
            .dt.total_days()
            .cast(pl.Int32)
            .alias('days_since_first_search'),
        ])
        .select(
            'user_id',
            'num_search',
            'days_since_last_search',
            'days_since_first_search',
        )
        .collect(streaming=True)
    )

    df = df.join(search_feat, on='user_id', how='left')
    del search_feat
    gc.collect()

    # Downcast remaining wide dtypes before pandas conversion
    int64_cols = [c for c, t in df.schema.items() if t == pl.Int64]
    float64_cols = [c for c, t in df.schema.items() if t == pl.Float64]
    cast_exprs = []
    if int64_cols:
        cast_exprs.extend(pl.col(c).cast(pl.Int32) for c in int64_cols)
    if float64_cols:
        cast_exprs.extend(pl.col(c).cast(pl.Float32) for c in float64_cols)
    if cast_exprs:
        df = df.with_columns(cast_exprs)

    return df

In [12]:
df = build_baseline_features(
    users_df=val_target,
    history_end_date=train_end_date,
    pred_start_date=val_start_date,
)

df

/tmp/ipykernel_22168/3562407109.py:50: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)
/tmp/ipykernel_22168/3562407109.py:86: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)


user_id,target,num_products_click,sum_discount_price_click,max_discount_price_click,days_since_last_click,days_since_first_click,num_products_favorite,sum_discount_price_favorite,max_discount_price_favorite,days_since_last_favorite,days_since_first_favorite,num_products_order,sum_discount_price_order,max_discount_price_order,days_since_last_order,days_since_first_order,num_products_to_cart,sum_discount_price_to_cart,max_discount_price_to_cart,days_since_last_to_cart,days_since_first_to_cart,num_search,days_since_last_search,days_since_first_search
i32,i8,i32,f32,f32,i32,i32,i32,f32,f32,i32,i32,i32,f32,f32,i32,i32,i32,f32,f32,i32,i32,i32,i32,i32
2422962,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
5975084,0,8,1720.0,344.0,35,111,16,7133.0,1961.0,20,95,null,null,null,null,null,77,14475.0,1817.0,4,108,20,4,111
10881532,1,18,22542.0,3411.0,18,106,1,0.0,null,18,18,42,10183.0,1183.0,18,92,47,13233.0,1213.0,18,119,100,1,119
7229058,1,206,56007.0,3399.0,0,117,5,1425.0,456.0,46,108,101,13984.0,596.0,46,117,126,17942.0,729.0,17,117,103,0,117
1666526,0,98,24368.0,840.0,5,95,14,3620.0,839.0,58,95,9,1398.0,399.0,5,5,29,4410.0,399.0,5,95,13,5,95
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3915700,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
7883746,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
8477981,0,15,47069.0,19990.0,5,108,2,5962.0,3189.0,108,108,37,6928.0,779.0,5,36,96,25012.0,980.0,5,118,166,5,119


In [13]:
feature_cols = [
    'num_products_click',
    'sum_discount_price_click', 'max_discount_price_click',
    'days_since_last_click', 'days_since_first_click',

    'num_products_favorite',
    'sum_discount_price_favorite', 'max_discount_price_favorite',
    'days_since_last_favorite', 'days_since_first_favorite',

    'num_products_order',
    'sum_discount_price_order', 'max_discount_price_order',
    'days_since_last_order', 'days_since_first_order',

    'num_products_to_cart',
    'sum_discount_price_to_cart', 'max_discount_price_to_cart',
    'days_since_last_to_cart', 'days_since_first_to_cart',

    'num_search',
    'days_since_last_search', 'days_since_first_search'
]

train_pl = df.filter((pl.col('user_id') % 10) <= 6)
eval_pl = df.filter((pl.col('user_id') % 10) > 6)

del df, val_target
gc.collect()

train_pd = train_pl.to_pandas()
eval_pd = eval_pl.to_pandas()

del train_pl, eval_pl
gc.collect()

for c in feature_cols:
    train_pd[c] = pd.to_numeric(train_pd[c], errors='coerce')
    eval_pd[c] = pd.to_numeric(eval_pd[c], errors='coerce')

train_pool = catboost.Pool(
    train_pd[feature_cols].reset_index(drop=True),
    label=train_pd['target'].reset_index(drop=True),
)

eval_pool = catboost.Pool(
    eval_pd[feature_cols].reset_index(drop=True),
    label=eval_pd['target'].reset_index(drop=True),
)

del train_pd, eval_pd
gc.collect()

0

In [14]:
train_pool.shape, eval_pool.shape

((1311636, 23), (563320, 23))

In [17]:
params = {
    'iterations': 200,
    'depth': 7,
    'learning_rate': 0.1,
    'random_state': 1,
    'eval_metric': 'AUC',
    'loss_function': 'Logloss',
    # 'task_type': 'GPU',
}

In [18]:
model = catboost.CatBoostClassifier(**params)
model.fit(
    train_pool,
    eval_set=eval_pool,
    use_best_model=True,
    verbose=10,
    early_stopping_rounds=50,
)

0:	test: 0.7434440	best: 0.7434440 (0)	total: 127ms	remaining: 25.2s
10:	test: 0.7524711	best: 0.7524711 (10)	total: 872ms	remaining: 15s
20:	test: 0.7548219	best: 0.7548219 (20)	total: 1.94s	remaining: 16.5s
30:	test: 0.7558153	best: 0.7558153 (30)	total: 2.64s	remaining: 14.4s
40:	test: 0.7564876	best: 0.7564876 (40)	total: 3.23s	remaining: 12.5s
50:	test: 0.7568172	best: 0.7568174 (49)	total: 3.82s	remaining: 11.2s
60:	test: 0.7571200	best: 0.7571200 (60)	total: 4.42s	remaining: 10.1s
70:	test: 0.7572900	best: 0.7572923 (69)	total: 5.01s	remaining: 9.11s
80:	test: 0.7574858	best: 0.7574858 (80)	total: 6.09s	remaining: 8.94s
90:	test: 0.7576156	best: 0.7576156 (90)	total: 6.97s	remaining: 8.35s
100:	test: 0.7577120	best: 0.7577120 (100)	total: 7.71s	remaining: 7.55s
110:	test: 0.7579136	best: 0.7579136 (110)	total: 8.51s	remaining: 6.83s
120:	test: 0.7580665	best: 0.7580665 (120)	total: 9.24s	remaining: 6.03s
130:	test: 0.7581623	best: 0.7581623 (130)	total: 9.99s	remaining: 5.26s
14

In [19]:
print("Best iteration:", model.best_iteration_)
print("Best score:", model.best_score_)
print("Trees in final model:", model.tree_count_)

Best iteration: 195
Best score: {'learn': {'Logloss': 0.5187545857073188}, 'validation': {'Logloss': 0.5202656671960901, 'AUC': 0.7587192906050458}}
Trees in final model: 196


In [20]:
name = 'baseline1'
model.save_model(f"{name}.bin")

In [21]:
fi = model.get_feature_importance(eval_pool, prettified=True)
fi.head(50)

,Feature Id,Importances
0,sum_discount_price_order,26.746245
1,days_since_last_order,11.415409
2,num_products_order,11.256307
3,days_since_first_order,7.499098
4,num_products_to_cart,6.765050
5,sum_discount_price_to_cart,6.390882
6,num_products_click,4.628145
7,days_since_last_to_cart,3.851959
8,num_search,3.731257
9,max_discount_price_to_cart,3.046506


In [22]:
test_users_submission = pl.read_csv(os.path.join(data_path, 'test_users.csv'))

df_test = build_baseline_features(
    users_df=test_users_submission,
    history_end_date=val_end_date,
    pred_start_date=test_start_date,
)

df_test_pd = df_test.to_pandas()

del df_test, test_users_submission
gc.collect()

for c in feature_cols:
    df_test_pd[c] = pd.to_numeric(df_test_pd[c], errors='coerce')

test_pool = catboost.Pool(df_test_pd[feature_cols])

/tmp/ipykernel_22168/3562407109.py:50: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)
/tmp/ipykernel_22168/3562407109.py:86: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)


In [23]:
df_test_pd['predict'] = model.predict(
    test_pool,
    prediction_type='Probability'
)[:, 1]

submission = df_test_pd[['user_id', 'predict']].copy()
submission

,user_id,predict
0,1342,0.164736
1,9852,0.792515
2,10206,0.221725
3,11317,0.219811
4,13289,0.600782
...,...,...
2068419,11157283,0.223263
2068420,11160395,0.147972
2068421,11165052,0.588919
2068422,11168218,0.521230


In [24]:
submission.to_csv('baseline1_submission.csv', index=False)